# NSTM only notebook (`nb-nstm-ppd.ipynb`)

Ce notebook est dedie a NSTM uniquement, mais il traite maintenant **tous les datasets** disponibles pour la comparaison:
- 20NG
- AGNews
- IMDB
- YahooAnswer

Le pipeline suit la logique de ton notebook ECTRM avec deux chemins de sortie:
- `Sortie_mat/output_os/NSTM`
- `Sortie_mat/output/kaggle/NSTM`

## Ce qui a ete adapte

Base utilisee:
- `Notebook/nb-ectrm-ppd.ipynb` pour la logique d'environnement et de sorties.
- `NSTM_source/NeuralSinkhornTopicModel` pour l'entrainement NSTM officiel.

Adaptations principales:
1. conversion automatique de tous les datasets vers le format `data.mat` attendu par NSTM,
2. boucle d'entrainement pour tous les datasets et tous les `K`,
3. export `.mat` normalise (`beta`, `train_theta`, `test_theta`) pour harmoniser la comparaison avec LDA/ECRTM,
4. generation metriques TD/Purity/NMI + fichiers topics pour Palmetto.

## Plan

1. Detecter l'environnement (LOCAL ou KAGGLE).
2. Detecter les datasets disponibles.
3. Verifier/charger le repo NSTM.
4. Convertir chaque dataset au format NSTM.
5. Entrainer NSTM pour chaque dataset et chaque `K`.
6. Sauvegarder les sorties dans `Sortie_mat`.
7. Calculer les metriques comparatives.

## 1) Imports

Cellule utilitaire avec imports standards et scientifiques.

In [2]:
import os
import sys
import glob
import json
import shutil
import subprocess
from pathlib import Path

import numpy as np
import scipy
import scipy.io
import scipy.sparse as sp

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("SciPy:", scipy.__version__)

Python: 3.13.1
NumPy: 2.2.1
SciPy: 1.17.0


## 2) Parametres globaux

Tous les hyperparametres du notebook sont centralises ici.

In [3]:
# Datasets cibles pour la comparaison
TARGET_DATASETS = ["20NG", "AGNews", "IMDB", "YahooAnswer"]

# Nombre de topics testes
K_LIST = [20, 50, 100]

# Reproductibilite
RANDOM_SEED = 42

# Hyperparametres NSTM (par defaut proches du repo original)
NSTM_EPOCHS = 50
NSTM_BATCH_SIZE = 200
NSTM_REC_LOSS_WEIGHT = 0.07
NSTM_SH_ALPHA = 20.0

# Comportement en cas d'erreur d'un run
CONTINUE_ON_ERROR = False

print("TARGET_DATASETS:", TARGET_DATASETS)
print("K_LIST:", K_LIST)

TARGET_DATASETS: ['20NG', 'AGNews', 'IMDB', 'YahooAnswer']
K_LIST: [20, 50, 100]


## 3) Detection environnement

On detecte si le notebook tourne en local ou sur Kaggle.

In [4]:
IS_KAGGLE = Path("/kaggle").exists()


def find_project_root(start):
    # Remonte les dossiers parents pour trouver la racine du projet local.
    for candidate in [start, *start.parents]:
        if (candidate / "NSTM_source").exists() and (candidate / "ECTRM_source").exists():
            return candidate
    return start


if IS_KAGGLE:
    PROJECT_ROOT = Path("/kaggle/working")
else:
    PROJECT_ROOT = find_project_root(Path.cwd())

print("IS_KAGGLE:", IS_KAGGLE)
print("PROJECT_ROOT:", PROJECT_ROOT)

IS_KAGGLE: False
PROJECT_ROOT: c:\Users\MLDSadmin\Desktop\PPD_2026


## 4) Chemins de sortie (double logique)

Sorties locales et Kaggle separentes, comme demande.

In [12]:
OUTPUT_OS_DIR = PROJECT_ROOT / "Sortie_mat" / "NSTM"
OUTPUT_KAGGLE_DIR = Path("/kaggle/working/Sortie_mat/output/kaggle/NSTM")

if IS_KAGGLE:
    OUTPUT_DIR = OUTPUT_KAGGLE_DIR
else:
    OUTPUT_DIR = OUTPUT_OS_DIR

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("OUTPUT_OS_DIR:", OUTPUT_OS_DIR)
print("OUTPUT_KAGGLE_DIR:", OUTPUT_KAGGLE_DIR)
print("OUTPUT_DIR (actif):", OUTPUT_DIR)

OUTPUT_OS_DIR: c:\Users\MLDSadmin\Desktop\PPD_2026\Sortie_mat\NSTM
OUTPUT_KAGGLE_DIR: \kaggle\working\Sortie_mat\output\kaggle\NSTM
OUTPUT_DIR (actif): c:\Users\MLDSadmin\Desktop\PPD_2026\Sortie_mat\NSTM


## 5) Detection des datasets disponibles

Le notebook cherche les datasets contenant tous les fichiers requis:
- `train_bow.npz`, `test_bow.npz`
- `train_labels.txt`, `test_labels.txt`
- `vocab.txt`, `word_embeddings.npz`

In [13]:
REQUIRED_FILES = [
    "train_bow.npz",
    "test_bow.npz",
    "train_labels.txt",
    "test_labels.txt",
    "vocab.txt",
    "word_embeddings.npz",
]

KNOWN_DATASETS = ["20NG", "AGNews", "IMDB", "YahooAnswer"]


def is_valid_dataset_dir(path):
    return path.exists() and path.is_dir() and all((path / f).exists() for f in REQUIRED_FILES)


def guess_dataset_name(path):
    s = str(path).lower()
    if "20ng" in s:
        return "20NG"
    if "agnews" in s or "ag-news" in s:
        return "AGNews"
    if "imdb" in s:
        return "IMDB"
    if "yahoo" in s:
        return "YahooAnswer"

    name = path.name.lower()
    if name == "20ng":
        return "20NG"
    if name == "agnews":
        return "AGNews"
    if name == "imdb":
        return "IMDB"
    if name == "yahooanswer":
        return "YahooAnswer"
    return None


def find_local_dataset_dirs(project_root):
    out = {}
    data_root = project_root / "ECTRM_source" / "ECRTM" / "data"

    # Priorite aux noms standards
    for ds in KNOWN_DATASETS:
        p = data_root / ds
        if is_valid_dataset_dir(p):
            out[ds] = p

    # Fallback: scanner tous les sous-dossiers
    if data_root.exists():
        for child in data_root.iterdir():
            if not child.is_dir():
                continue
            if is_valid_dataset_dir(child):
                ds = guess_dataset_name(child) or child.name
                out.setdefault(ds, child)

    return out


def find_kaggle_dataset_dirs():
    out = {}

    explicit_candidates = {
        "20NG": [
            "/kaggle/input/kaggleinputectrm-source-data-20ng",
            "/kaggle/input/ectrm-source-data-20ng",
            "/kaggle/input/ectrm-source-data-20ng/20NG",
        ],
        "AGNews": [
            "/kaggle/input/ectrm-sourcedataagnews",
            "/kaggle/input/ectrm-source-data-agnews",
            "/kaggle/input/ectrm-source-data-agnews/AGNews",
        ],
        "IMDB": [
            "/kaggle/input/ectrm-sourcedataimdb",
            "/kaggle/input/ectrm-source-data-imdb",
            "/kaggle/input/ectrm-source-data-imdb/IMDB",
        ],
        "YahooAnswer": [
            "/kaggle/input/ectrm-sourcedatayahooanswer",
            "/kaggle/input/ectrm-source-data-yahooanswer",
            "/kaggle/input/ectrm-source-data-yahooanswer/YahooAnswer",
        ],
    }

    for ds, paths in explicit_candidates.items():
        for c in paths:
            p = Path(c)
            if is_valid_dataset_dir(p):
                out[ds] = p
                break

    # Fallback: scan simple /kaggle/input/* et /kaggle/input/*/*
    input_root = Path("/kaggle/input")
    if input_root.exists():
        for first_level in input_root.iterdir():
            if not first_level.is_dir():
                continue

            to_check = [first_level]
            to_check.extend([x for x in first_level.iterdir() if x.is_dir()])

            for candidate in to_check:
                if not is_valid_dataset_dir(candidate):
                    continue
                ds = guess_dataset_name(candidate)
                if ds is not None and ds not in out:
                    out[ds] = candidate

    return out


if IS_KAGGLE:
    discovered = find_kaggle_dataset_dirs()
else:
    discovered = find_local_dataset_dirs(PROJECT_ROOT)

# Filtre selon TARGET_DATASETS
DATASET_DIRS = {ds: discovered[ds] for ds in TARGET_DATASETS if ds in discovered}
missing = [ds for ds in TARGET_DATASETS if ds not in DATASET_DIRS]

if not DATASET_DIRS:
    raise RuntimeError("Aucun dataset valide detecte pour NSTM")

print("Datasets detectes:")
for ds, p in DATASET_DIRS.items():
    print(f" - {ds}: {p}")

if missing:
    print("Datasets manquants:", missing)

Datasets detectes:
 - 20NG: c:\Users\MLDSadmin\Desktop\PPD_2026\ECTRM_source\data\20NG
 - AGNews: c:\Users\MLDSadmin\Desktop\PPD_2026\ECTRM_source\data\AGNews
 - IMDB: c:\Users\MLDSadmin\Desktop\PPD_2026\ECTRM_source\data\IMDB
 - YahooAnswer: c:\Users\MLDSadmin\Desktop\PPD_2026\ECTRM_source\data\YahooAnswer


## 6) Resolution du repo NSTM

Source attendue:
- local: `NSTM_source/NeuralSinkhornTopicModel`
- Kaggle: copie vers `/kaggle/working` pour pouvoir ecrire dans `save/`

In [14]:
def looks_like_nstm_repo(path):
    return (
        path.exists()
        and (path / "nstm.py").exists()
        and (path / "utils.py").exists()
        and (path / "auto_diff_sinkhorn.py").exists()
    )


def find_nstm_repo_under(root):
    if not root.exists():
        return None
    for nstm_file in root.rglob("nstm.py"):
        candidate = nstm_file.parent
        if looks_like_nstm_repo(candidate):
            return candidate
    return None


local_repo = PROJECT_ROOT / "NSTM_source"
kaggle_work_repo = Path("/kaggle/working/NSTM_source")

if IS_KAGGLE:
    if looks_like_nstm_repo(kaggle_work_repo):
        NSTM_REPO = kaggle_work_repo
    else:
        found = find_nstm_repo_under(Path("/kaggle/input"))
        if found is not None:
            kaggle_work_repo.parent.mkdir(parents=True, exist_ok=True)
            if kaggle_work_repo.exists():
                shutil.rmtree(kaggle_work_repo)
            shutil.copytree(found, kaggle_work_repo)
            NSTM_REPO = kaggle_work_repo
        else:
            # Fallback clone
            kaggle_work_repo.parent.mkdir(parents=True, exist_ok=True)
            cmd = [
                "git",
                "clone",
                "https://github.com/ethanhezhao/NeuralSinkhornTopicModel",
                str(kaggle_work_repo),
            ]
            print("Cloning:", " ".join(cmd))
            subprocess.run(cmd, check=True)
            NSTM_REPO = kaggle_work_repo
else:
    if not looks_like_nstm_repo(local_repo):
        raise FileNotFoundError(f"Repo NSTM introuvable: {local_repo}")
    NSTM_REPO = local_repo

print("NSTM_REPO:", NSTM_REPO)

NSTM_REPO: c:\Users\MLDSadmin\Desktop\PPD_2026\NSTM_source\NeuralSinkhornTopicModel


## 7) Verification des versions de librairies

Point important:
- le repo NSTM officiel est base sur TensorFlow 1.x,
- il peut necessiter des adaptations si ton runtime est en TensorFlow 2.x.

Cette cellule verifie l'etat actuel et laisse des options explicites.

In [15]:
# Optionnel: installer les requirements exacts du repo NSTM (peut casser un runtime moderne)
INSTALL_NSTM_REQUIREMENTS = False

if INSTALL_NSTM_REQUIREMENTS:
    req = NSTM_REPO / "requirements.txt"
    cmd = [sys.executable, "-m", "pip", "install", "-r", str(req)]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)
else:
    print("Skip requirements install (INSTALL_NSTM_REQUIREMENTS=False)")


# Check TensorFlow
try:
    import tensorflow as tf
    print("TensorFlow:", tf.__version__)

    # Detection de type de runtime
    if tf.__version__.startswith("1."):
        print("TF1 runtime detecte: compatible avec le code officiel NSTM")
    else:
        print("TF2 runtime detecte: le code officiel NSTM peut necessiter patch/adaptation")

    try:
        gpus = tf.config.list_physical_devices("GPU")
        print("GPUs:", gpus)
    except Exception:
        print("GPU check via tf.config non disponible")
except Exception as exc:
    print("TensorFlow import error:", repr(exc))

Skip requirements install (INSTALL_NSTM_REQUIREMENTS=False)
TensorFlow import error: ModuleNotFoundError("No module named 'tensorflow'")


## 8) Patch TF2 (optionnel)

Si ton environnement n'a pas TF1 mais TF2, active ce patch pour tenter une execution en mode compatibilite.

Par defaut, il est inactif pour conserver le code source original.

In [16]:
APPLY_TF2_COMPAT_PATCH = False


def patch_file_text(path, transform_func):
    txt = path.read_text(encoding="utf-8")
    new_txt = transform_func(txt)
    if new_txt != txt:
        path.write_text(new_txt, encoding="utf-8")
        return True
    return False


def apply_tf2_patch(repo_root):
    nstm_path = repo_root / "nstm.py"
    utils_path = repo_root / "utils.py"
    sinkhorn_path = repo_root / "auto_diff_sinkhorn.py"

    def patch_import_block(txt):
        marker = "# PPD_TF2_PATCH"
        if marker in txt:
            return txt
        txt = txt.replace(
            "import tensorflow as tf",
            "# PPD_TF2_PATCH\nimport tensorflow.compat.v1 as tf\ntf.disable_v2_behavior()",
            1,
        )
        return txt

    def patch_batch_norm(txt):
        old = "doc_topic_tf = tf.contrib.layers.batch_norm(utils.linear(doc_topic_tf, FLAGS.K, scope='mean'))"
        if old in txt:
            new = "mean_linear = utils.linear(doc_topic_tf, FLAGS.K, scope='mean')\n        mean, var = tf.nn.moments(mean_linear, axes=[0])\n        doc_topic_tf = tf.nn.batch_normalization(mean_linear, mean, var, offset=None, scale=None, variance_epsilon=1e-5)"
            txt = txt.replace(old, new, 1)
        return txt

    changed = []
    changed.append((nstm_path, patch_file_text(nstm_path, lambda t: patch_batch_norm(patch_import_block(t)))))
    changed.append((utils_path, patch_file_text(utils_path, patch_import_block)))
    changed.append((sinkhorn_path, patch_file_text(sinkhorn_path, patch_import_block)))

    return changed


if APPLY_TF2_COMPAT_PATCH:
    changed = apply_tf2_patch(NSTM_REPO)
    print("TF2 patch applied status:")
    for p, ok in changed:
        print(" -", p.name, "changed=" + str(ok))
else:
    print("Skip TF2 patch (APPLY_TF2_COMPAT_PATCH=False)")

Skip TF2 patch (APPLY_TF2_COMPAT_PATCH=False)


## 9) Conversion de tous les datasets au format NSTM

Le repo NSTM attend un fichier `datasets/<name>/data.mat` contenant:
- `wordsTrain` (V x Ntrain)
- `wordsTest` (V x Ntest)
- `labelsTrain`, `labelsTest`
- `labelsToGroup`
- `vocabulary`
- `embeddings`

In [17]:
DATASET_CACHE = {}

for dataset, data_dir in DATASET_DIRS.items():
    print("\n=== Preparing dataset:", dataset, "===")

    train_bow = sp.load_npz(data_dir / "train_bow.npz").astype(np.float32)
    test_bow = sp.load_npz(data_dir / "test_bow.npz").astype(np.float32)

    train_labels = np.loadtxt(data_dir / "train_labels.txt", dtype=np.int32)
    test_labels = np.loadtxt(data_dir / "test_labels.txt", dtype=np.int32)

    with open(data_dir / "vocab.txt", "r", encoding="utf-8") as f:
        vocab = [line.strip() for line in f if line.strip()]

    emb_data = np.load(data_dir / "word_embeddings.npz")
    if isinstance(emb_data, np.lib.npyio.NpzFile):
        emb_flat = emb_data[list(emb_data.keys())[0]]
    else:
        emb_flat = emb_data

    emb_flat = emb_flat.astype(np.float32)
    V = train_bow.shape[1]
    emb_dim = emb_flat.size // V
    embeddings = emb_flat[: V * emb_dim].reshape(V, emb_dim).astype(np.float32)

    words_train = train_bow.transpose().tocsr()
    words_test = test_bow.transpose().tocsr()

    unique_labels = sorted(np.unique(np.concatenate([train_labels, test_labels])))

    labels_to_group = np.empty((len(unique_labels), 1), dtype=object)
    for i, label_id in enumerate(unique_labels):
        labels_to_group[i, 0] = np.array([f"label_{int(label_id)}"], dtype=object)

    vocabulary_obj = np.empty((len(vocab), 1), dtype=object)
    for i, token in enumerate(vocab):
        vocabulary_obj[i, 0] = np.array([token], dtype=object)

    nstm_dataset_name = f"PPD_{dataset}"
    nstm_dataset_dir = NSTM_REPO / "datasets" / nstm_dataset_name
    nstm_dataset_dir.mkdir(parents=True, exist_ok=True)
    mat_path = nstm_dataset_dir / "data.mat"

    scipy.io.savemat(
        str(mat_path),
        {
            "wordsTrain": words_train,
            "wordsTest": words_test,
            "labelsTrain": train_labels.reshape(-1, 1),
            "labelsTest": test_labels.reshape(-1, 1),
            "labelsToGroup": labels_to_group,
            "vocabulary": vocabulary_obj,
            "embeddings": embeddings,
        },
    )

    DATASET_CACHE[dataset] = {
        "data_dir": data_dir,
        "nstm_dataset_name": nstm_dataset_name,
        "nstm_data_mat": mat_path,
        "vocab": vocab,
        "test_labels": test_labels,
        "n_train": int(train_bow.shape[0]),
        "n_test": int(test_bow.shape[0]),
        "vocab_size": int(V),
    }

    print("Saved:", mat_path)
    print("train docs:", train_bow.shape[0], "| test docs:", test_bow.shape[0], "| vocab:", V)


=== Preparing dataset: 20NG ===
Saved: c:\Users\MLDSadmin\Desktop\PPD_2026\NSTM_source\NeuralSinkhornTopicModel\datasets\PPD_20NG\data.mat
train docs: 11314 | test docs: 7532 | vocab: 5000

=== Preparing dataset: AGNews ===
Saved: c:\Users\MLDSadmin\Desktop\PPD_2026\NSTM_source\NeuralSinkhornTopicModel\datasets\PPD_AGNews\data.mat
train docs: 10000 | test docs: 2500 | vocab: 5000

=== Preparing dataset: IMDB ===
Saved: c:\Users\MLDSadmin\Desktop\PPD_2026\NSTM_source\NeuralSinkhornTopicModel\datasets\PPD_IMDB\data.mat
train docs: 25000 | test docs: 25000 | vocab: 5000

=== Preparing dataset: YahooAnswer ===
Saved: c:\Users\MLDSadmin\Desktop\PPD_2026\NSTM_source\NeuralSinkhornTopicModel\datasets\PPD_YahooAnswer\data.mat
train docs: 10000 | test docs: 2500 | vocab: 5000


## 10) Entrainement NSTM pour tous les datasets et tous les K

Cette cellule lance NSTM en boucle:
- dataset dans `TARGET_DATASETS` detectes
- K dans `K_LIST`

In [18]:
RUN_REGISTRY = {}
all_save_pattern = str(NSTM_REPO / "save" / "**" / "save.mat")

for dataset in DATASET_CACHE.keys():
    RUN_REGISTRY[dataset] = {}
    nstm_dataset_name = DATASET_CACHE[dataset]["nstm_dataset_name"]

    for K in K_LIST:
        try:
            before = set(glob.glob(all_save_pattern, recursive=True))

            cmd = [
                sys.executable,
                "nstm.py",
                f"--dataset={nstm_dataset_name}",
                f"--K={K}",
                f"--random_seed={RANDOM_SEED}",
                f"--n_epochs={NSTM_EPOCHS}",
                f"--batch_size={NSTM_BATCH_SIZE}",
                f"--rec_loss_weight={NSTM_REC_LOSS_WEIGHT}",
                f"--sh_alpha={NSTM_SH_ALPHA}",
            ]

            print("\nRunning:", " ".join(cmd))
            subprocess.run(cmd, cwd=str(NSTM_REPO), check=True)

            after = set(glob.glob(all_save_pattern, recursive=True))
            new_files = sorted(after - before, key=lambda p: os.path.getmtime(p))

            if new_files:
                run_mat = Path(new_files[-1])
            else:
                pattern = (
                    NSTM_REPO
                    / "save"
                    / f"dataset{nstm_dataset_name}_K{K}_RW*_RS{RANDOM_SEED}_L*"
                    / "save.mat"
                )
                candidates = sorted(glob.glob(str(pattern)), key=lambda p: os.path.getmtime(p))
                if not candidates:
                    raise FileNotFoundError(f"save.mat introuvable pour dataset={dataset}, K={K}")
                run_mat = Path(candidates[-1])

            RUN_REGISTRY[dataset][K] = {
                "run_mat": run_mat,
                "run_dir": run_mat.parent,
            }

            print("Saved run mat:", run_mat)

        except Exception as exc:
            print(f"ERROR dataset={dataset} K={K}: {exc}")
            if not CONTINUE_ON_ERROR:
                raise


Running: c:\Users\MLDSadmin\AppData\Local\Programs\Python\Python313\python.exe nstm.py --dataset=PPD_20NG --K=20 --random_seed=42 --n_epochs=50 --batch_size=200 --rec_loss_weight=0.07 --sh_alpha=20.0
ERROR dataset=20NG K=20: Command '['c:\\Users\\MLDSadmin\\AppData\\Local\\Programs\\Python\\Python313\\python.exe', 'nstm.py', '--dataset=PPD_20NG', '--K=20', '--random_seed=42', '--n_epochs=50', '--batch_size=200', '--rec_loss_weight=0.07', '--sh_alpha=20.0']' returned non-zero exit status 1.


CalledProcessError: Command '['c:\\Users\\MLDSadmin\\AppData\\Local\\Programs\\Python\\Python313\\python.exe', 'nstm.py', '--dataset=PPD_20NG', '--K=20', '--random_seed=42', '--n_epochs=50', '--batch_size=200', '--rec_loss_weight=0.07', '--sh_alpha=20.0']' returned non-zero exit status 1.

## 11) Normalisation des sorties `.mat`

Sortie NSTM officielle: `phi`, `train_theta`, `test_theta`.

Sortie harmonisee pour comparaison:
- `beta = phi`
- `train_theta`
- `test_theta`

In [ ]:
NORMALIZED_REGISTRY = {}

for dataset, k_map in RUN_REGISTRY.items():
    dataset_out_dir = OUTPUT_DIR / dataset
    dataset_out_dir.mkdir(parents=True, exist_ok=True)
    NORMALIZED_REGISTRY[dataset] = {}

    for K, meta in k_map.items():
        run_mat = meta["run_mat"]
        loaded = scipy.io.loadmat(str(run_mat))

        if "phi" not in loaded:
            raise KeyError(f"'phi' introuvable dans {run_mat}")
        if "train_theta" not in loaded or "test_theta" not in loaded:
            raise KeyError(f"theta introuvable dans {run_mat}")

        beta = loaded["phi"].astype(np.float32)
        train_theta = loaded["train_theta"].astype(np.float32)
        test_theta = loaded["test_theta"].astype(np.float32)

        normalized_path = dataset_out_dir / f"{dataset}_NSTM_K{K}_seed{RANDOM_SEED}.mat"
        scipy.io.savemat(
            str(normalized_path),
            {
                "beta": beta,
                "train_theta": train_theta,
                "test_theta": test_theta,
            "traintheta": train_theta,
            "testtheta": test_theta,
            },
        )

        raw_copy = dataset_out_dir / f"{dataset}_NSTM_raw_K{K}_seed{RANDOM_SEED}.mat"
        shutil.copy2(run_mat, raw_copy)

        NORMALIZED_REGISTRY[dataset][K] = normalized_path

        print("Saved normalized:", normalized_path)

## 12) Metriques NSTM (TD, Purity, NMI)

Calcule les metriques par dataset et par K.

In [ ]:
import pandas as pd
from sklearn.metrics import normalized_mutual_info_score


def topic_diversity_from_beta(beta, vocab_list, topn=25):
    all_words = []
    for k in range(beta.shape[0]):
        top_idx = np.argsort(beta[k])[::-1][:topn]
        all_words.extend(vocab_list[i] for i in top_idx)
    return len(set(all_words)) / max(1, len(all_words))


def purity_score(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    total = 0
    for c in np.unique(y_pred):
        idx = np.where(y_pred == c)[0]
        if len(idx) == 0:
            continue
        _, counts = np.unique(y_true[idx], return_counts=True)
        total += counts.max()
    return total / len(y_true)


rows = []
for dataset, k_map in NORMALIZED_REGISTRY.items():
    vocab = DATASET_CACHE[dataset]["vocab"]
    y_true_all = DATASET_CACHE[dataset]["test_labels"]

    for K, mat_path in k_map.items():
        loaded = scipy.io.loadmat(str(mat_path))
        beta = loaded["beta"]
        test_theta = loaded["test_theta"]

        n_eval = min(len(y_true_all), test_theta.shape[0])
        y_true = y_true_all[:n_eval]
        y_pred = test_theta[:n_eval].argmax(axis=1)

        td = topic_diversity_from_beta(beta, vocab, topn=25)
        purity = purity_score(y_true, y_pred)
        nmi = normalized_mutual_info_score(y_true, y_pred)

        rows.append(
            {
                "dataset": dataset,
                "model": "NSTM",
                "K": int(K),
                "seed": int(RANDOM_SEED),
                "TD": round(float(td), 4),
                "Purity": round(float(purity), 4),
                "NMI": round(float(nmi), 4),
                "n_eval_docs": int(n_eval),
            }
        )

metrics_df = pd.DataFrame(rows).sort_values(["dataset", "K"]) if rows else pd.DataFrame()

# CSV global
global_csv = OUTPUT_DIR / "NSTM_TD_Purity_NMI_ALL.csv"
metrics_df.to_csv(global_csv, index=False)
print("Saved:", global_csv)

# CSV par dataset
if not metrics_df.empty:
    for dataset in sorted(metrics_df["dataset"].unique()):
        ds_csv = OUTPUT_DIR / dataset / f"NSTM_TD_Purity_NMI_{dataset}.csv"
        metrics_df[metrics_df["dataset"] == dataset].to_csv(ds_csv, index=False)
        print("Saved:", ds_csv)

print("\nMetrics preview:")
if not metrics_df.empty:
    print(metrics_df.to_string(index=False))
else:
    print("No metrics computed")

## 13) Export des topics texte pour Palmetto

Genere un fichier `topics_for_cv_...txt` par dataset et par K.

In [ ]:
TOPN = 15

for dataset, k_map in NORMALIZED_REGISTRY.items():
    dataset_out_dir = OUTPUT_DIR / dataset
    vocab = DATASET_CACHE[dataset]["vocab"]

    for K, mat_path in k_map.items():
        loaded = scipy.io.loadmat(str(mat_path))
        beta = loaded["beta"]

        topics_file = dataset_out_dir / f"topics_for_cv_{dataset}_NSTM_K{K}_seed{RANDOM_SEED}.txt"
        with open(topics_file, "w", encoding="utf-8") as f:
            for k in range(beta.shape[0]):
                top_idx = np.argsort(beta[k])[::-1][:TOPN]
                f.write(" ".join(vocab[i] for i in top_idx) + "\n")

        print("Saved:", topics_file)

## 14) Recap des sorties

Rappel:
- local: `Sortie_mat/output_os/NSTM/<dataset>/...`
- Kaggle: `Sortie_mat/output/kaggle/NSTM/<dataset>/...`

In [ ]:
print("OUTPUT_DIR actif:", OUTPUT_DIR)
print("\nFichiers generes:")

for path in sorted(OUTPUT_DIR.rglob("*")):
    if path.is_file():
        rel = path.relative_to(OUTPUT_DIR)
        size_kb = path.stat().st_size / 1024.0
        print(f" - {rel} ({size_kb:.1f} KB)")